# Genie Benchmark Runner

This notebook runs a Genie space against its configured benchmark questions across **N independent runs**
and records accuracy results in MLflow.

### What it does

1. Pulls the Genie space configuration (benchmark questions + expected SQL) by space ID
2. Pre-executes every expected SQL against the warehouse and caches the result datasets
3. Runs every benchmark question through the live Genie API **NUM_RUNS** times
4. Executes each generated SQL and compares the resulting dataset against the expected dataset
5. Outputs a results DataFrame with: run index, question, generated SQL, and pass/fail
6. Creates an MLflow run logging overall accuracy (avg across all runs) and per-run accuracy

### Pass / Fail Logic

| Condition | Result |
|-----------|--------|
| Expected and generated datasets match (same shape + values) | PASS |
| Datasets differ in row count, column count, or values | FAIL |
| Benchmark has no expected SQL and a SQL was generated | PASS |
| No SQL was generated (error or empty response) | FAIL |
| Expected SQL failed to execute (bad benchmark) | SKIP |

### Dataset Comparison Rules

- **Row ordering is ignored** — both result sets are sorted before comparison
- **Column names are ignored** — only column count and cell values matter (handles aliasing)
- **Numeric tolerance** — float-parseable values are compared with relative tolerance (~1e-6)

In [ ]:
%pip install databricks-sdk langchain langchain-databricks langchain-core mlflow typing-extensions -qU

dbutils.library.restartPython()

In [ ]:
import json
import time
import pandas as pd
from datetime import datetime

import mlflow
from databricks.sdk import WorkspaceClient

## Configuration

Set your Genie space ID and tuning parameters below.

When running inside a Databricks notebook, `WorkspaceClient()` authenticates automatically.
For local execution, set the `DATABRICKS_HOST` and `DATABRICKS_TOKEN` environment variables.

In [ ]:
GENIE_SPACE_ID  = "<your-genie-space-id>"
NUM_RUNS        = 5      # number of complete benchmark passes to execute
MAX_WAIT_SEC    = 180    # per-question timeout in seconds
ROW_LIMIT       = 1000   # max rows fetched per SQL execution (safety cap)
NUMERIC_RTOL    = 1e-6   # relative tolerance for floating-point comparison
MLFLOW_EXPERIMENT_NAME = "/genie-benchmark-runner"

## Load Genie Space & Benchmark Questions

Fetch the full space definition from the Databricks Genie API. Benchmark questions live
under `benchmarks.questions` in the serialized space config. Each question includes an
`answer` array with expected SQL that we execute and compare against.

The space's `warehouse_id` is used for all SQL execution via the Statement Execution API.

In [ ]:
w = WorkspaceClient()

space = w.genie.get_space(
    space_id=GENIE_SPACE_ID,
    include_serialized_space=True,
)

space_config        = json.loads(space.serialized_space)
benchmark_questions = space_config.get("benchmarks", {}).get("questions", [])
warehouse_id        = space.warehouse_id

# Normalise each benchmark to {question, expected_sql}
benchmarks = []
for q in benchmark_questions:
    question_text = " ".join(q.get("question", [])).strip()
    expected_sql = None
    for answer in q.get("answer", []):
        if answer.get("format") == "SQL" and answer.get("content"):
            expected_sql = "".join(answer["content"]).strip()
            break
    if question_text:
        benchmarks.append({"question": question_text, "expected_sql": expected_sql})

print(f"Space:               {space.title}")
print(f"Description:         {space.description or 'N/A'}")
print(f"Warehouse:           {warehouse_id}")
print(f"Benchmark questions: {len(benchmarks)}")
print(f"Runs planned:        {NUM_RUNS}")
print(f"Total API calls:     {len(benchmarks) * NUM_RUNS}")

print("\n--- Benchmark Questions ---")
for i, b in enumerate(benchmarks, 1):
    print(f"  {i:>2}. {b['question']}")
    if b["expected_sql"]:
        for line in b["expected_sql"].splitlines():
            print(f"        {line}")
    else:
        print(f"        (no expected SQL)")

## Helper Functions

### `run_genie_question`
Starts a new Genie conversation for a question and polls until the response is complete,
then extracts the generated SQL from the message attachments.

### `execute_sql`
Runs a SQL statement against the warehouse via the Statement Execution API and returns
the result as a pandas DataFrame.

### `compare_datasets`
Compares two DataFrames for equivalence — ignoring column names and row ordering, with
numeric tolerance for floating-point values.

### `evaluate_pass`
Orchestrates the evaluation: executes the generated SQL, then compares its result dataset
against the cached expected dataset.

In [ ]:
def run_genie_question(w, space_id, question_text, max_wait_seconds=MAX_WAIT_SEC):
    """
    Submit a question to a Genie space and return (generated_sql, error_message).

    Returns
    -------
    generated_sql : str | None
        The SQL query generated by Genie, or None on failure.
    error : str | None
        A description of what went wrong, or None on success.
    """
    try:
        resp = w.genie.start_conversation(space_id=space_id, content=question_text)

        if hasattr(resp, "conversation_id") and hasattr(resp, "message_id"):
            conv_id = resp.conversation_id
            msg_id  = resp.message_id
        elif hasattr(resp, "conversation") and hasattr(resp, "message"):
            conv_id = resp.conversation.id
            msg_id  = resp.message.id
        else:
            return None, f"Unexpected start_conversation response shape: {type(resp)}"

        deadline = time.time() + max_wait_seconds
        msg = None
        while time.time() < deadline:
            msg = w.genie.get_message(
                space_id=space_id,
                conversation_id=conv_id,
                message_id=msg_id,
            )
            status = msg.status.value if hasattr(msg.status, "value") else str(msg.status)
            status_upper = status.upper()

            if status_upper == "COMPLETED":
                for att in (getattr(msg, "attachments", None) or []):
                    query_part = getattr(att, "query", None)
                    if query_part:
                        sql = getattr(query_part, "query", None)
                        if sql:
                            return sql.strip(), None
                return None, "Completed but no SQL found in message attachments"

            elif status_upper in ("FAILED", "ERROR", "CANCELLED", "QUERY_RESULT_EXPIRED"):
                return None, f"Message ended with status: {status}"

            time.sleep(3)

        return None, f"Timeout after {max_wait_seconds}s — last status: {status}"

    except Exception as exc:
        return None, f"{type(exc).__name__}: {exc}"

In [ ]:
def _get_state(resp):
    """Extract the status state string from a StatementResponse."""
    state = resp.status.state
    if hasattr(state, "value"):
        state = state.value
    return str(state).upper()


def _get_error(resp):
    """Extract an error message from a StatementResponse, if any."""
    err = getattr(resp.status, "error", None)
    if err is None:
        return ""
    return getattr(err, "message", str(err))


def _response_to_df(resp):
    """Convert a successful StatementResponse into a pandas DataFrame."""
    columns = [col.name for col in resp.manifest.schema.columns]
    rows = resp.result.data_array if resp.result and resp.result.data_array else []
    return pd.DataFrame(rows, columns=columns)


def execute_sql(w, warehouse_id, sql, row_limit=ROW_LIMIT, max_wait=MAX_WAIT_SEC):
    """
    Execute a SQL statement via the Statement Execution API and return the
    result as a pandas DataFrame.

    Returns
    -------
    df : pd.DataFrame | None
    error : str | None
    """
    try:
        resp = w.statement_execution.execute_statement(
            warehouse_id=warehouse_id,
            statement=sql,
            wait_timeout="50s",
            row_limit=row_limit,
        )

        state = _get_state(resp)

        if state == "SUCCEEDED":
            return _response_to_df(resp), None

        if state in ("FAILED", "CANCELED", "CLOSED"):
            return None, f"Statement {state}: {_get_error(resp)}"

        # Still running — poll until done
        deadline = time.time() + max_wait
        while time.time() < deadline:
            time.sleep(2)
            resp = w.statement_execution.get_statement(resp.statement_id)
            state = _get_state(resp)
            if state == "SUCCEEDED":
                return _response_to_df(resp), None
            if state in ("FAILED", "CANCELED", "CLOSED"):
                return None, f"Statement {state}: {_get_error(resp)}"

        try:
            w.statement_execution.cancel_execution(resp.statement_id)
        except Exception:
            pass
        return None, f"Statement timed out after {max_wait}s"

    except Exception as exc:
        return None, f"{type(exc).__name__}: {exc}"


def _is_null(val):
    if val is None:
        return True
    try:
        return pd.isna(val)
    except (ValueError, TypeError):
        return False


def compare_datasets(expected_df, generated_df, rtol=NUMERIC_RTOL):
    """
    Compare two DataFrames for equivalence, ignoring column names and row order.

    Returns
    -------
    passed : bool
    reasoning : str
    """
    if expected_df.shape[1] != generated_df.shape[1]:
        return False, (
            f"Column count mismatch: expected {expected_df.shape[1]}, "
            f"got {generated_df.shape[1]}"
        )

    if len(expected_df) != len(generated_df):
        return False, (
            f"Row count mismatch: expected {len(expected_df)}, "
            f"got {len(generated_df)}"
        )

    if expected_df.empty:
        return True, "Both result sets are empty"

    # Normalise column names to positional indices so sorting is consistent
    col_indices = list(range(expected_df.shape[1]))
    exp = expected_df.copy()
    gen = generated_df.copy()
    exp.columns = col_indices
    gen.columns = col_indices

    # Sort by all columns (as strings) to eliminate row-order differences
    exp = exp.sort_values(by=col_indices, key=lambda s: s.astype(str)).reset_index(drop=True)
    gen = gen.sort_values(by=col_indices, key=lambda s: s.astype(str)).reset_index(drop=True)

    # Cell-by-cell comparison
    mismatches = []
    for row_idx in range(len(exp)):
        for col_idx in col_indices:
            ev = exp.iloc[row_idx, col_idx]
            gv = gen.iloc[row_idx, col_idx]

            if _is_null(ev) and _is_null(gv):
                continue
            if _is_null(ev) != _is_null(gv):
                mismatches.append((row_idx, col_idx, ev, gv))
                continue

            if str(ev) == str(gv):
                continue

            # Numeric tolerance check
            try:
                ef, gf = float(ev), float(gv)
                if abs(ef - gf) <= rtol * max(abs(ef), abs(gf), 1.0):
                    continue
            except (ValueError, TypeError):
                pass

            mismatches.append((row_idx, col_idx, ev, gv))

    if not mismatches:
        return True, f"Datasets match ({len(expected_df)} rows, {expected_df.shape[1]} cols)"

    first = mismatches[0]
    return False, (
        f"{len(mismatches)} value mismatch(es); "
        f"first at row {first[0]}, col {first[1]}: "
        f"expected '{first[2]}', got '{first[3]}'"
    )


def evaluate_pass(w, warehouse_id, expected_sql, expected_df, generated_sql, row_limit=ROW_LIMIT):
    """
    Determine whether a generated SQL passes the benchmark by comparing
    the result datasets.

    Parameters
    ----------
    expected_sql : str | None
    expected_df  : pd.DataFrame | None   (pre-executed expected result)
    generated_sql : str | None

    Returns
    -------
    passed : bool
    reasoning : str
    """
    if not generated_sql:
        return False, "No SQL was generated"

    if not expected_sql:
        return True, "SQL generated (no expected SQL to compare against)"

    if expected_df is None:
        return False, "SKIP — expected SQL failed to execute (bad benchmark)"

    generated_df, exec_error = execute_sql(w, warehouse_id, generated_sql, row_limit=row_limit)
    if exec_error:
        return False, f"Generated SQL failed to execute: {exec_error}"

    return compare_datasets(expected_df, generated_df)

## Pre-execute Expected SQL

Run every expected SQL statement once against the warehouse and cache the resulting
DataFrames. These cached results are reused across all benchmark runs, avoiding
redundant warehouse calls.

Benchmarks whose expected SQL fails to execute are flagged — they indicate a problem
with the benchmark definition, not with Genie.

In [ ]:
if not benchmarks:
    raise ValueError(
        f"No benchmark questions found in space '{GENIE_SPACE_ID}'. "
        "Add benchmark questions to the Genie space before running this notebook."
    )

print("Pre-executing expected SQL for each benchmark...\n")

valid_count = 0
skip_count  = 0
no_sql_count = 0

for i, b in enumerate(benchmarks, 1):
    if not b["expected_sql"]:
        b["expected_df"] = None
        no_sql_count += 1
        print(f"  {i:>2}. (no expected SQL) — {b['question'][:70]}")
        continue

    df, error = execute_sql(w, warehouse_id, b["expected_sql"])
    b["expected_df"] = df

    if error:
        skip_count += 1
        print(f"  {i:>2}. [SKIP] {error[:80]}")
        print(f"        {b['question'][:70]}")
    else:
        valid_count += 1
        print(f"  {i:>2}. [OK]   {df.shape[0]} rows, {df.shape[1]} cols — {b['question'][:60]}")

print(f"\nPre-execution complete: {valid_count} valid, {skip_count} failed, {no_sql_count} without SQL")

## Run Benchmark Executions

Execute all benchmark questions **NUM_RUNS** times. Each run sends every question to the
live Genie API, executes the generated SQL against the warehouse, and compares the
resulting dataset against the pre-cached expected dataset.

> **Note:** With 20 benchmark questions and 5 runs, this makes 100 Genie API calls
> plus up to 100 SQL executions. Allow ~5–15 minutes depending on query complexity
> and warehouse warm-up time.

In [ ]:
raw_results = []

for run_idx in range(1, NUM_RUNS + 1):
    print(f"\n{'='*60}")
    print(f"  Run {run_idx} / {NUM_RUNS}")
    print(f"{'='*60}")

    run_pass  = 0
    run_total = len(benchmarks)

    for q_idx, benchmark in enumerate(benchmarks, 1):
        question     = benchmark["question"]
        expected_sql = benchmark["expected_sql"]
        expected_df  = benchmark.get("expected_df")

        print(f"  [{run_idx}/{NUM_RUNS}] Q{q_idx:>2}/{run_total} — {question[:80]}..." if len(question) > 80
              else f"  [{run_idx}/{NUM_RUNS}] Q{q_idx:>2}/{run_total} — {question}")

        generated_sql, genie_error = run_genie_question(w, GENIE_SPACE_ID, question)

        passed, reasoning = evaluate_pass(
            w, warehouse_id, expected_sql, expected_df, generated_sql, ROW_LIMIT
        )

        status_icon = "PASS" if passed else "FAIL"
        print(f"          -> [{status_icon}] {reasoning[:100]}")
        if genie_error:
            print(f"          -> [ERROR] {genie_error}")

        if passed:
            run_pass += 1

        raw_results.append({
            "run_index":     run_idx,
            "question":      question,
            "expected_sql":  expected_sql,
            "generated_sql": generated_sql,
            "passed":        passed,
            "reasoning":     reasoning,
            "error":         genie_error,
        })

    run_pct = run_pass / run_total * 100
    print(f"\n  Run {run_idx} accuracy: {run_pass}/{run_total}  ({run_pct:.1f}%)")

print("\nAll runs complete.")

## Results DataFrame

Full results table with one row per (run, question). Columns:

| Column | Description |
|--------|-------------|
| `run_index` | Which run (1–NUM_RUNS) |
| `question` | The benchmark question text |
| `expected_sql` | Expected SQL from the Genie space config |
| `generated_sql` | SQL produced by the live Genie API |
| `passed` | Whether the generated dataset matched the expected dataset |
| `reasoning` | Dataset comparison result (shape, match, or mismatch details) |
| `error` | Any Genie API error (None on success) |

In [ ]:
results_df = pd.DataFrame(raw_results)

results_df = results_df[[
    "run_index", "question", "expected_sql", "generated_sql",
    "passed", "reasoning", "error",
]]

print(f"Total rows: {len(results_df)}  ({NUM_RUNS} runs \u00d7 {len(benchmarks)} questions)")

try:
    display(results_df)  # noqa: F821
except NameError:
    print(results_df.to_string())

## Summary Statistics

Per-run accuracy and overall accuracy averaged across all runs.

In [ ]:
per_run_accuracy = (
    results_df.groupby("run_index")["passed"]
    .agg(passed_count="sum", total="count")
    .assign(accuracy_pct=lambda df: df["passed_count"] / df["total"] * 100)
    .reset_index()
)

overall_accuracy = per_run_accuracy["accuracy_pct"].mean()

print(f"Space:            {space.title}")
print(f"Benchmark count:  {len(benchmarks)}")
print(f"Runs:             {NUM_RUNS}")
print()
print("Per-run accuracy:")
for _, row in per_run_accuracy.iterrows():
    bar = '\u2588' * int(row['accuracy_pct'] / 5) + '\u2591' * (20 - int(row['accuracy_pct'] / 5))
    print(f"  Run {int(row['run_index'])}: {bar} {row['accuracy_pct']:5.1f}%  "
          f"({int(row['passed_count'])}/{int(row['total'])})")

bar = '\u2588' * int(overall_accuracy / 5) + '\u2591' * (20 - int(overall_accuracy / 5))
print(f"\n  Overall: {bar} {overall_accuracy:5.1f}%")

try:
    display(per_run_accuracy)  # noqa: F821
except NameError:
    print(per_run_accuracy.to_string())

## MLflow Run

Log all results to MLflow:

| Logged Item | Type | Description |
|-------------|------|-------------|
| `space_id` | param | Genie space ID |
| `space_name` | param | Human-readable space name |
| `num_benchmarks` | param | Number of benchmark questions |
| `num_runs` | param | Number of runs executed |
| `row_limit` | param | Row limit applied to SQL executions |
| `evaluation_method` | param | `dataset_comparison` |
| `overall_accuracy` | metric | Average accuracy across all runs (%) |
| `run_N_accuracy` | metric | Per-run accuracy (%) for runs 1–N |
| `benchmark_results.csv` | artifact | Full results DataFrame |

In [ ]:
import tempfile
import os

run_timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
run_name = f"{space.title.replace(' ', '_')}_{run_timestamp}"

mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)

with mlflow.start_run(run_name=run_name) as mlflow_run:

    mlflow.log_params({
        "space_id":          GENIE_SPACE_ID,
        "space_name":        space.title,
        "num_benchmarks":    len(benchmarks),
        "num_runs":          NUM_RUNS,
        "row_limit":         ROW_LIMIT,
        "evaluation_method": "dataset_comparison",
    })

    for _, row in per_run_accuracy.iterrows():
        run_num = int(row["run_index"])
        mlflow.log_metric(f"run_{run_num}_accuracy", round(row["accuracy_pct"], 2))

    mlflow.log_metric("overall_accuracy", round(overall_accuracy, 2))

    with tempfile.TemporaryDirectory() as tmpdir:
        csv_path = os.path.join(tmpdir, "benchmark_results.csv")
        results_df.to_csv(csv_path, index=False)
        mlflow.log_artifact(csv_path)

    run_id  = mlflow_run.info.run_id

print(f"MLflow run complete.")
print(f"  Run name:         {run_name}")
print(f"  Run ID:           {run_id}")
print(f"  Experiment:       {MLFLOW_EXPERIMENT_NAME}")
print(f"  Overall accuracy: {overall_accuracy:.1f}%")
print()
print("Per-run metrics logged:")
for _, row in per_run_accuracy.iterrows():
    print(f"  run_{int(row['run_index'])}_accuracy = {row['accuracy_pct']:.2f}%")
print(f"  overall_accuracy = {overall_accuracy:.2f}%")

## Benchmark Analysis & Recommendations

Analyse the benchmark results to identify failure patterns and generate actionable
recommendations for improving Genie space accuracy.

The analysis computes per-question pass rates across runs, categorises failure types,
and feeds everything into an LLM chain that produces a structured improvement plan.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_databricks import ChatDatabricks

# ------------------------------------------------------------------
# 1. Compute per-question statistics
# ------------------------------------------------------------------
question_stats = (
    results_df.groupby("question")
    .agg(
        total_runs=("passed", "count"),
        passes=("passed", "sum"),
        failures=("passed", lambda s: (~s).sum()),
    )
    .assign(pass_rate=lambda df: (df["passes"] / df["total_runs"] * 100).round(1))
    .sort_values("pass_rate")
    .reset_index()
)

# ------------------------------------------------------------------
# 2. Categorise failure reasons
# ------------------------------------------------------------------
failed_rows = results_df[~results_df["passed"]].copy()

def categorise(row):
    r = row["reasoning"] or ""
    if "No SQL was generated" in r:
        return "no_sql_generated"
    if "failed to execute" in r:
        return "execution_error"
    if "Row count mismatch" in r:
        return "row_count_mismatch"
    if "Column count mismatch" in r:
        return "column_count_mismatch"
    if "value mismatch" in r:
        return "value_mismatch"
    if "SKIP" in r:
        return "bad_benchmark"
    return "other"

failed_rows["failure_category"] = failed_rows.apply(categorise, axis=1)

failure_summary = (
    failed_rows.groupby("failure_category")
    .size()
    .sort_values(ascending=False)
    .to_dict()
)

# ------------------------------------------------------------------
# 3. Build per-question failure detail for the LLM
# ------------------------------------------------------------------
question_details = []
for _, row in question_stats.iterrows():
    q = row["question"]
    q_failures = failed_rows[failed_rows["question"] == q]
    reasons = q_failures["reasoning"].dropna().unique().tolist()
    sample_expected = results_df.loc[results_df["question"] == q, "expected_sql"].iloc[0]
    sample_generated = q_failures["generated_sql"].dropna().head(1).tolist()

    detail = f"  Q: {q}\n"
    detail += f"     Pass rate: {row['pass_rate']}% ({int(row['passes'])}/{int(row['total_runs'])})\n"
    if sample_expected:
        detail += f"     Expected SQL: {sample_expected[:200]}\n"
    if sample_generated:
        detail += f"     Sample generated SQL: {sample_generated[0][:200]}\n"
    if reasons:
        detail += f"     Failure reasons: {'; '.join(r[:120] for r in reasons[:3])}\n"
    question_details.append(detail)

per_question_text = "\n".join(question_details)

# ------------------------------------------------------------------
# 4. Identify consistently vs intermittently failing questions
# ------------------------------------------------------------------
always_fail   = question_stats[question_stats["pass_rate"] == 0]["question"].tolist()
sometimes_fail = question_stats[
    (question_stats["pass_rate"] > 0) & (question_stats["pass_rate"] < 100)
]["question"].tolist()
always_pass   = question_stats[question_stats["pass_rate"] == 100]["question"].tolist()

consistency_text = (
    f"Always pass ({len(always_pass)}): {', '.join(always_pass[:5])}"
    + (" ..." if len(always_pass) > 5 else "") + "\n"
    f"Intermittent ({len(sometimes_fail)}): {', '.join(sometimes_fail[:5])}"
    + (" ..." if len(sometimes_fail) > 5 else "") + "\n"
    f"Always fail ({len(always_fail)}): {', '.join(always_fail[:5])}"
    + (" ..." if len(always_fail) > 5 else "")
)

# ------------------------------------------------------------------
# 5. Print diagnostic summary
# ------------------------------------------------------------------
print("=== Failure Category Breakdown ===")
for cat, count in failure_summary.items():
    print(f"  {cat:30s} {count}")
print(f"\n  Total failures: {len(failed_rows)} / {len(results_df)}")

print(f"\n=== Question Consistency ===")
print(f"  Always pass:    {len(always_pass)}")
print(f"  Intermittent:   {len(sometimes_fail)}")
print(f"  Always fail:    {len(always_fail)}")

In [ ]:
LLM_ENDPOINT = "databricks-meta-llama-3-1-70b-instruct"

llm = ChatDatabricks(endpoint=LLM_ENDPOINT, temperature=0.0)

analysis_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are an expert at optimising Databricks AI/BI Genie spaces. "
     "You will be given benchmark results from a Genie space and your job is to "
     "analyse the failure patterns and produce specific, actionable recommendations "
     "to improve the space's accuracy. Focus on changes the space author can actually "
     "make: adding instructions, example SQL, table/column metadata, join specs, "
     "SQL snippets, or rewording benchmark questions. Be concise and specific."),
    ("human", """Analyse the following benchmark results and provide recommendations.

**Space:** {space_name}
**Description:** {space_description}
**Overall accuracy:** {overall_accuracy:.1f}%
**Runs:** {num_runs}
**Benchmarks:** {num_benchmarks}

---

### Failure Category Breakdown
{failure_categories}

### Question Consistency
{consistency}

### Per-Question Detail (sorted by pass rate, worst first)
{question_details}

---

Provide your analysis as a structured report with:

1. **Executive Summary** — 2-3 sentence overview of benchmark health and the most critical issues

2. **Root Cause Analysis** — for each major failure pattern, explain the likely cause:
   - Why are certain questions always failing?
   - Why do intermittent failures occur?
   - What do the failure categories (row count mismatch, value mismatch, etc.) reveal?

3. **Recommendations** — 5-8 specific, prioritised actions the space author should take.
   For each recommendation:
   - What to change (e.g., "Add an example SQL for...", "Add a text instruction that...")
   - Why it will help
   - Priority: HIGH / MEDIUM / LOW

4. **Quick Wins** — 2-3 changes that are easy to implement and likely to have immediate impact""")
])

analysis_chain = analysis_prompt | llm | StrOutputParser()

failure_categories_text = "\n".join(
    f"  {cat}: {count}" for cat, count in failure_summary.items()
) if failure_summary else "  (no failures)"

analysis_report = analysis_chain.invoke({
    "space_name": space.title,
    "space_description": space.description or "No description provided",
    "overall_accuracy": overall_accuracy,
    "num_runs": NUM_RUNS,
    "num_benchmarks": len(benchmarks),
    "failure_categories": failure_categories_text,
    "consistency": consistency_text,
    "question_details": per_question_text,
})

print(analysis_report)